# Phase 15 — Temperature Variation: Detectability vs T + Diversity Ablation

**Item 6 of the Jun 17, 2026 advisor meeting.** Qwen2.5-Math-7B-Instruct on MATH-500.

**Q1 — Does temperature change detectability?** Single-pass L-SML continuous (GOOD_5)
AUROC at each T ∈ {0.3, 0.6, 1.0, 1.5, 2.0}.

**Q2 — Diversity vs more passes (primary).** Paired ablation on the same samples:

| Condition | Passes | Question |
|---|---|---|
| **A** | K=5, all at T=1.0 | more passes alone |
| **B** | K=5, one per temperature | temperature diversity |

Both conditions share the T=1.0 run0 base pass and predict its correctness label,
so `paired_boot_delta_auc` gives a paired test of AUC(B) − AUC(A).

**Run matrix** (9 generation runs, one model load):
T ∈ {0.3, 0.6, 1.5, 2.0} × run0 + T=1.0 × run0…run4. ~5–9 A100-hours, fully resumable.

**Dual purpose**: every run saves the full raw-data schema (`token_entropies`,
`token_spilled_energies`, `top_k_logprobs`, `gen_token_ids`, `full_text`, `label`,
`question`). T=1.0 run0 thereby becomes the canonical raw-trace cache for the
MATH-500/Qwen-7B cell — repaying the Step-148 Extension E data debt (no raw traces
existed anywhere for this cell).

**Pre-registered gates** (Cell — Gates):
- **G-T1**: some T ≠ 1.0 with AUROC ≥ AUROC(T=1.0) + 2pp and non-overlapping 95% CIs
  (unpaired — labels differ per T).
- **G-T2 (primary)**: Δ = AUC(B) − AUC(A) ≥ +2pp with paired 95% CI excluding 0 →
  temperature diversity matters; |Δ| < 2pp → it's just more passes.


In [ ]:
# Cell 1 — Clone + pip install + imports
import os, sys, shutil

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

REPO_DIR = '/content/hallucination_detection'
# Branch clone: generate_full(top_k_logprobs=...), paired_boot_delta_auc and
# multipass_lsml_continuous live on this branch until the careful merge to
# master (master stays stable while Phase 12 Corrected runs against it).
BRANCH = 'experiment/item6-temperature'

if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')):
    shutil.rmtree(REPO_DIR)

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {BRANCH} https://github.com/omrisegev/hallucination_detection.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} fetch -q origin {BRANCH}')
    os.system(f'git -C {REPO_DIR} checkout -q {BRANCH}')
    os.system(f'git -C {REPO_DIR} pull -q')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.system('pip install -q "transformers>=4.40" accelerate datasets bitsandbytes autoawq scipy scikit-learn')

from spectral_utils import (
    load_model, generate_full, free_memory,
    extract_all_features, sw_var_peak_with_window, FEAT_NAMES,
    load_cache, save_cache,
    load_math500, math_prompt, is_correct_math,
    zscore, boot_auc, paired_boot_delta_auc,
    lsml_continuous_pipeline, multipass_lsml_continuous, anchor_orient,
)
import numpy as np, pickle
from tqdm.auto import tqdm

import datasets  # noqa: F401 — freeze pyarrow in memory before any gptqmodel install
print('spectral_utils imported OK')

In [ ]:
# Cell 2 — Config
MODEL_ID   = 'Qwen/Qwen2.5-Math-7B-Instruct'   # matches Phase 12 Corrected Section 2
N_SAMPLES  = 200
MAX_NEW    = 2048    # reasoning traces — SAME cap at every T (Phase 12 Corrected MAX_NEW_RSN)
TOP_K_SAVE = 50      # raw-data rule: top-50 logprobs per token (compact numpy schema)
SAVE_EVERY = 25

TEMPS = [0.3, 0.6, 1.0, 1.5, 2.0]
# (T, run_idx): run0 per T = Q1 curve + Condition B view; T=1.0 runs 1-4
# complete Condition A (K=5 same-temperature passes).
RUN_SPECS = [(0.3, 0), (0.6, 0), (1.0, 0), (1.0, 1), (1.0, 2), (1.0, 3), (1.0, 4), (1.5, 0), (2.0, 0)]

CACHE_DIR = '/content/drive/MyDrive/hallucination_detection/cache/phase15_temperature'
RES_DIR   = os.path.join(CACHE_DIR, 'results')

def run_path(T, r):
    return os.path.join(CACHE_DIR, f'math500_qwen7b_T{T}_run{r}.pkl')

# ── L-SML config (Step 134 GOOD_5 + continuous pipeline) ─────────────────
GOOD_FEATURES  = ['epr', 'low_band_power', 'sw_var_peak', 'cusum_max', 'spectral_entropy']
FEATURE_SIGNS  = {f: -1 for f in GOOD_FEATURES}  # higher value → more likely wrong
ANCHOR_FEATURE = 'epr'   # label-free global-sign anchor (Step 148 anchor_orient fix)

# ── Pre-registered gates ──────────────────────────────────────────────────
GATE_T1_PP = 0.02   # G-T1: some T≠1.0 beats T=1.0 by ≥2pp, non-overlapping CIs
GATE_T2_PP = 0.02   # G-T2 (primary): AUC(B)−AUC(A) ≥ +2pp, paired CI excludes 0

A_SPECS = [(1.0, 0), (1.0, 1), (1.0, 2), (1.0, 3), (1.0, 4)]   # Condition A
B_SPECS = [(0.3, 0), (0.6, 0), (1.0, 0), (1.5, 0), (2.0, 0)]   # Condition B

print(f'Config OK: {MODEL_ID}, N={N_SAMPLES}, {len(RUN_SPECS)} runs, MAX_NEW={MAX_NEW}')

In [ ]:
# Cell 3 — Mount Google Drive + create cache dirs + provenance probe
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

# Provenance probe — INFORMATIONAL ONLY. Phase 12 Corrected p2 holds T=1.0
# Qwen-Math-7B entropies, but its early samples predate the Step-149/150
# grading fixes and it has no top-k logprobs, so Phase 15 never reuses it.
P12_P2 = '/content/drive/MyDrive/hallucination_detection/cache/phase12_corrected/p2_math500_qwenmath7b.pkl'
if os.path.exists(P12_P2):
    _p2 = load_cache(P12_P2)
    _done = sum(bool(v.get('done')) for v in _p2.values())
    print(f'phase12_corrected p2 present: {_done} done — NOT reused (label provenance + no logprobs)')
else:
    print('phase12_corrected p2 not found (fine — not reused anyway)')

for T, r in RUN_SPECS:
    c = load_cache(run_path(T, r))
    d = sum(bool(v.get('done')) for v in c.values())
    print(f'  T={T} run{r}: {d}/{N_SAMPLES} done')
print(f'Cache dir ready: {CACHE_DIR}')

## Inference — 9 runs, one model load

Per-run pkl on Drive, checkpoint every 25 samples — resumable across sessions
at both the run level and the sample level.

In [ ]:
# Cell 4 — Load model (flat Drive cache — Drive FUSE breaks HF symlinks)
from huggingface_hub import snapshot_download

FLAT_CACHE = '/content/drive/MyDrive/hf_cache_flat'
os.makedirs(FLAT_CACHE, exist_ok=True)

def ensure_flat_dir(repo_id):
    local_dir = os.path.join(FLAT_CACHE, repo_id.replace('/', '__'))
    if os.path.exists(os.path.join(local_dir, 'config.json')):
        print(f'Using cached: {local_dir}')
        return local_dir
    print(f'Downloading {repo_id} to flat dir...')
    try:
        snapshot_download(repo_id=repo_id, local_dir=local_dir, local_dir_use_symlinks=False)
    except TypeError:
        snapshot_download(repo_id=repo_id, local_dir=local_dir)
    return local_dir

mdl, tok = load_model(ensure_flat_dir(MODEL_ID), quantize_4bit=False)
print(f'Model loaded: {MODEL_ID}')

In [ ]:
# Cell 5 — Inference driver: 9 runs × N_SAMPLES, full raw-data schema
# Dual purpose: (1) Item 6 temperature sweep; (2) T=1.0 run0 becomes the
# canonical raw-trace cache for MATH-500/Qwen-7B (Step-148 Extension E debt).
math_rows = load_math500(n_samples=N_SAMPLES)
print(f'Dataset: {len(math_rows)} samples')

for T, r in RUN_SPECS:
    path  = run_path(T, r)
    cache = load_cache(path)
    done0 = sum(bool(v.get('done')) for v in cache.values())
    if done0 >= len(math_rows):
        print(f'T={T} run{r}: complete ({done0}) — skipping')
        continue
    print(f'T={T} run{r}: resuming at {done0}/{len(math_rows)}')
    for i, row in enumerate(tqdm(math_rows, desc=f'T={T} run{r}')):
        if cache.get(i, {}).get('done'):
            continue
        res = generate_full(mdl, tok, math_prompt(row), temperature=T,
                            max_new_tokens=MAX_NEW, top_k_logprobs=TOP_K_SAVE)
        cache[i] = {
            'done':                   True,
            'question':               row.get('problem', row.get('query', row.get('question', ''))),
            'full_text':              res['full_text'],
            'token_entropies':        res['token_entropies'],
            'token_spilled_energies': res['token_spilled_energies'],
            'top_k_logprobs':         res['top_k_logprobs'],
            'gen_token_ids':          res['gen_token_ids'],
            'label':                  int(is_correct_math(res['full_text'], row)),
        }
        if (i + 1) % SAVE_EVERY == 0:
            save_cache(cache, path)
    save_cache(cache, path)
    acc = np.mean([v['label'] for v in cache.values() if v.get('done')])
    print(f'T={T} run{r}: complete — accuracy {acc:.1%}')

In [ ]:
# Cell 6 — Unload model (analysis below is CPU-only)
del mdl, tok
free_memory()
print('Model unloaded')

## Analysis — CPU only

Every cell persists its output to Drive (`background_save` survival pattern);
safe to re-run after a disconnect without the model.

In [ ]:
# Cell 7 — Feature extraction (persisted) + common-valid intersection
FEATS_PATH = os.path.join(RES_DIR, 'phase15_feats.pkl')
FORCE_RECOMPUTE = False

def _feats_valid(runs):
    # Stale-pkl guard: a feats pkl written while inference was incomplete has
    # missing or empty specs. (Recompute from the raw caches is cheap CPU.)
    return isinstance(runs, dict) and set(runs) == set(RUN_SPECS) \
        and all(len(v) > 0 for v in runs.values())

if not FORCE_RECOMPUTE and 'RUNS' in globals() and _feats_valid(RUNS):
    print('already in memory; skipping')
else:
    RUNS = {}
    if not FORCE_RECOMPUTE and os.path.exists(FEATS_PATH):
        _cand = pickle.load(open(FEATS_PATH, 'rb'))
        if _feats_valid(_cand):
            RUNS = _cand
            print(f'loaded from {FEATS_PATH}')
        else:
            print('STALE/PARTIAL feats pkl detected — recomputing from raw caches')
    if not RUNS:
        for T, r in RUN_SPECS:
            cache = load_cache(run_path(T, r))
            per = {}
            for i in range(N_SAMPLES):
                e = cache.get(i, {})
                if not e.get('done'):
                    continue
                f = extract_all_features(e['token_entropies'], e.get('token_spilled_energies'))
                if f is not None:   # None = trace too short for spectral features
                    per[i] = {'feats': f, 'label': int(e['label'])}
            RUNS[(T, r)] = per
        with open(FEATS_PATH, 'wb') as fh:
            pickle.dump(RUNS, fh)
        print(f'saved to {FEATS_PATH}')

for T, r in RUN_SPECS:
    print(f'T={T} run{r}: {len(RUNS[(T, r)])}/{N_SAMPLES} valid')

# Condition A vs B must score the exact same samples → intersect valid
# indices over ALL 9 runs (T=2.0 degenerate traces shrink this — report it).
COMMON_IDX = sorted(set.intersection(*[set(RUNS[s]) for s in RUN_SPECS]))
print(f'Common-valid intersection: {len(COMMON_IDX)}/{N_SAMPLES} samples')

def feats_on(spec, idx):
    return {f: np.array([RUNS[spec][i]['feats'][f] for i in idx]) for f in GOOD_FEATURES}

def labels_on(spec, idx):
    return np.array([RUNS[spec][i]['label'] for i in idx])

In [ ]:
# Cell 8 — sw_var_peak window ablation (T=1.0 run0, standard cell)
WINDOW_SIZES = [8, 16, 24, 32, 48]
spec10  = (1.0, 0)
idx10   = sorted(RUNS[spec10])
y10     = labels_on(spec10, idx10)
cache10 = load_cache(run_path(*spec10))
for w in WINDOW_SIZES:
    vals = np.array([sw_var_peak_with_window(cache10[i]['token_entropies'], w) for i in idx10])
    auc, lo, hi = boot_auc(y10, -vals)   # sign −1: higher variance peak → wrong
    print(f'sw_var_peak w={w:>2}: AUROC {auc:.3f} [{lo:.3f}, {hi:.3f}]')

In [ ]:
# Cell 9 — Individual feature AUROC per temperature (each T on its own run0)
FEAT_TABLE = {}
header = f'{"feature":<18}' + ''.join(f'  T={T:<5}' for T in TEMPS)
print(header)
print('-' * len(header))
for f in GOOD_FEATURES:
    row = f'{f:<18}'
    for T in TEMPS:
        spec = (T, 0)
        idx  = sorted(RUNS[spec])
        y    = labels_on(spec, idx)
        v    = np.array([RUNS[spec][i]['feats'][f] for i in idx])
        auc, _, _ = boot_auc(y, v * FEATURE_SIGNS[f])
        FEAT_TABLE[(f, T)] = auc
        row += f'  {auc:.3f} '
    print(row)

In [ ]:
# Cell 10a — Q1: single-pass L-SML continuous AUROC vs temperature
Q1 = {}
for T in TEMPS:
    spec = (T, 0)
    idx  = sorted(RUNS[spec])
    y    = labels_on(spec, idx)
    mp   = multipass_lsml_continuous([feats_on(spec, idx)], GOOD_FEATURES,
                                     FEATURE_SIGNS, anchor_feature=ANCHOR_FEATURE)
    auc, lo, hi = boot_auc(y, mp['fused'])
    minority = int(min(y.sum(), len(y) - y.sum()))
    Q1[T] = {'auc': auc, 'lo': lo, 'hi': hi, 'n': len(idx), 'acc': float(y.mean()),
             'minority': minority, 'scores': mp['fused'], 'labels': y}
    warn = '  WARNING: minority class < 10 — underpowered' if minority < 10 else ''
    print(f'T={T}: AUROC {auc:.3f} [{lo:.3f}, {hi:.3f}]  acc={y.mean():.1%}  n={len(idx)}{warn}')

In [ ]:
# Cell 10b — Q2 (primary): K=5 same-T (A) vs K=5 multi-T (B), paired on the
# common-valid samples. Labels = T=1.0 run0 correctness (the shared base pass
# present in BOTH conditions), so A and B predict the same label vector.
yC = labels_on((1.0, 0), COMMON_IDX)
print(f'Common samples: {len(yC)}, base-pass accuracy {yC.mean():.1%}')

A_runs = [feats_on(s, COMMON_IDX) for s in A_SPECS]
B_runs = [feats_on(s, COMMON_IDX) for s in B_SPECS]

mpA = multipass_lsml_continuous(A_runs, GOOD_FEATURES, FEATURE_SIGNS, anchor_feature=ANCHOR_FEATURE)
mpB = multipass_lsml_continuous(B_runs, GOOD_FEATURES, FEATURE_SIGNS, anchor_feature=ANCHOR_FEATURE)
mp1 = multipass_lsml_continuous([feats_on((1.0, 0), COMMON_IDX)], GOOD_FEATURES,
                                FEATURE_SIGNS, anchor_feature=ANCHOR_FEATURE)

rows = [
    ('single pass T=1.0 (base)',      mp1['fused']),
    ('A: K=5 same-T, L-SML',          mpA['fused']),
    ('A: K=5 same-T, simple avg',     mpA['avg_fused']),
    ('B: K=5 multi-T, L-SML',         mpB['fused']),
    ('B: K=5 multi-T, simple avg',    mpB['avg_fused']),
]
print(f'\n{"Method":<32} {"AUROC":>7} {"95% CI":>17}')
print('-' * 60)
Q2_AUCS = {}
for name, sc in rows:
    auc, lo, hi = boot_auc(yC, sc)
    Q2_AUCS[name] = (auc, lo, hi)
    print(f'{name:<32} {auc:>7.3f}  [{lo:.3f}, {hi:.3f}]')

# Paired deltas — the actual tests
D_BA = paired_boot_delta_auc(yC, mpB['fused'], mpA['fused'])   # diversity effect
D_A1 = paired_boot_delta_auc(yC, mpA['fused'], mp1['fused'])   # more-passes effect
print(f'\npaired AUC(B) − AUC(A):    {D_BA[0]:+.3f}  [{D_BA[1]:+.3f}, {D_BA[2]:+.3f}]')
print(f'paired AUC(A) − AUC(base): {D_A1[0]:+.3f}  [{D_A1[1]:+.3f}, {D_A1[2]:+.3f}]')

# Cross-pass dependence — A's off-diagonals ≥ B's is itself the diversity diagnostic
def show_rho(name, rho):
    off = rho[np.triu_indices_from(rho, 1)]
    print(f'\n{name} per-pass Spearman rho (off-diag mean {off.mean():+.2f}):')
    for row in rho:
        print('   ' + ' '.join(f'{v:+.2f}' for v in row))
    hi_pairs = [(i, j) for i in range(len(rho)) for j in range(i + 1, len(rho))
                if abs(rho[i, j]) >= 0.75]
    if hi_pairs:
        print(f'   pairs with |rho| >= 0.75: {hi_pairs}')
show_rho('Condition A', mpA['rho'])
show_rho('Condition B', mpB['rho'])

In [ ]:
# Cell 10c — Robustness (secondary): flat 25-view fusion (5 features × 5 passes).
# Expect clustering to group by feature (the within-pass correlation block is
# replicated 5x), which is why the hierarchical variant above is primary.
def flat_views(specs):
    d = {}
    for k, s in enumerate(specs):
        fd = feats_on(s, COMMON_IDX)
        for name in GOOD_FEATURES:
            d[f'{name}@p{k}'] = fd[name]
    return d

signs25 = {f'{name}@p{k}': FEATURE_SIGNS[name] for name in GOOD_FEATURES for k in range(5)}
names25 = list(signs25)
FLAT25 = {}
for cond, specs, mp in (('A', A_SPECS, mpA), ('B', B_SPECS, mpB)):
    sc, meta = lsml_continuous_pipeline(flat_views(specs), names25, signs25)
    sc, _ = anchor_orient(sc, mp['avg_fused'])   # same label-free anchor policy
    auc, lo, hi = boot_auc(yC, sc)
    FLAT25[cond] = (auc, lo, hi)
    print(f'25-view flat {cond}: AUROC {auc:.3f} [{lo:.3f}, {hi:.3f}]  K={meta.get("K", "?")}')

## Gates, results, plots

In [ ]:
# Cell 11 — Pre-registered gate verdicts
print('=' * 70)
base = Q1[1.0]
g1_hits = [T for T in TEMPS if T != 1.0
           and Q1[T]['auc'] >= base['auc'] + GATE_T1_PP
           and Q1[T]['lo'] > base['hi']]
G_T1 = bool(g1_hits)
print(f"G-T1 (some T beats T=1.0 by >={GATE_T1_PP*100:.0f}pp, non-overlapping CIs): "
      f"{'PASS' if G_T1 else 'FAIL'}"
      + (f' — {g1_hits}' if g1_hits else ''))
print('      caveat: unpaired — each T is scored against its own labels')

d, lo, hi = D_BA
G_T2 = (d >= GATE_T2_PP) and (lo > 0)
print(f"G-T2 PRIMARY (AUC(B)-AUC(A) >= +{GATE_T2_PP*100:.0f}pp, paired CI excludes 0): "
      f"{'PASS' if G_T2 else 'FAIL'}  ({d:+.3f} [{lo:+.3f}, {hi:+.3f}])")
if G_T2:
    print('      → temperature DIVERSITY is the source of the multi-pass lift')
elif abs(d) < GATE_T2_PP:
    print('      → A ≈ B: the lift (if any) comes from more passes, not T-diversity')
else:
    print('      → direction/CI inconclusive at pilot n — see paired CI')
da, dalo, dahi = D_A1
print(f'Context: AUC(A) - AUC(single pass) = {da:+.3f} [{dalo:+.3f}, {dahi:+.3f}] '
      '(does K=5 same-T fusion help at all)')
print('=' * 70)

In [ ]:
# Cell 12 — Save consolidated results dict (raw scores kept — derive later)
RESULTS_PATH = os.path.join(RES_DIR, 'phase15_results.pkl')
results = {
    'config': {'model': MODEL_ID, 'n_samples': N_SAMPLES, 'max_new': MAX_NEW,
               'temps': TEMPS, 'run_specs': RUN_SPECS, 'features': GOOD_FEATURES,
               'signs': FEATURE_SIGNS, 'anchor': ANCHOR_FEATURE,
               'gate_t1_pp': GATE_T1_PP, 'gate_t2_pp': GATE_T2_PP},
    'q1': Q1,                               # per-T auc/ci/acc/scores/labels
    'feat_table': FEAT_TABLE,               # (feature, T) → auc
    'q2': {'labels': yC, 'common_idx': COMMON_IDX, 'aucs': Q2_AUCS,
           'A_fused': mpA['fused'], 'A_avg': mpA['avg_fused'], 'A_rho': mpA['rho'],
           'B_fused': mpB['fused'], 'B_avg': mpB['avg_fused'], 'B_rho': mpB['rho'],
           'base_fused': mp1['fused'],
           'delta_B_minus_A': D_BA, 'delta_A_minus_base': D_A1,
           'flat25': FLAT25,
           'A_meta': mpA['meta'], 'B_meta': mpB['meta']},
    'gates': {'G_T1': G_T1, 'G_T1_hits': g1_hits, 'G_T2': G_T2},
}
with open(RESULTS_PATH, 'wb') as f:
    pickle.dump(results, f)
print(f'Saved to {RESULTS_PATH}')

In [ ]:
# Cell 13 — Plots
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (1) AUROC vs T with CI band + accuracy overlay
ax = axes[0, 0]
aucs = [Q1[T]['auc'] for T in TEMPS]
los  = [Q1[T]['lo'] for T in TEMPS]
his  = [Q1[T]['hi'] for T in TEMPS]
ax.plot(TEMPS, aucs, 'o-', color='tab:blue', label='L-SML cont. (GOOD_5)')
ax.fill_between(TEMPS, los, his, alpha=0.2, color='tab:blue')
ax.axhline(0.5, color='gray', ls=':', lw=1)
ax2 = ax.twinx()
ax2.plot(TEMPS, [Q1[T]['acc'] for T in TEMPS], 's--', color='tab:orange', label='accuracy')
ax2.set_ylabel('accuracy', color='tab:orange')
ax.set_xlabel('temperature'); ax.set_ylabel('AUROC', color='tab:blue')
ax.set_title('Q1: detectability vs temperature (single pass)')
ax.legend(loc='lower left')

# (2) feature × T heatmap
ax = axes[0, 1]
M = np.array([[FEAT_TABLE[(f, T)] for T in TEMPS] for f in GOOD_FEATURES])
im = ax.imshow(M, cmap='RdYlGn', vmin=0.4, vmax=0.9, aspect='auto')
ax.set_xticks(range(len(TEMPS)), [f'T={T}' for T in TEMPS])
ax.set_yticks(range(len(GOOD_FEATURES)), GOOD_FEATURES)
for a in range(len(GOOD_FEATURES)):
    for b in range(len(TEMPS)):
        ax.text(b, a, f'{M[a, b]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Individual feature AUROC')

# (3) Condition A vs B bars with paired delta
ax = axes[1, 0]
names = list(Q2_AUCS)
vals  = [Q2_AUCS[n][0] for n in names]
errs  = [(Q2_AUCS[n][0] - Q2_AUCS[n][1], Q2_AUCS[n][2] - Q2_AUCS[n][0]) for n in names]
colors = ['gray', 'tab:blue', 'lightsteelblue', 'tab:red', 'lightcoral']
ax.bar(range(len(names)), vals, yerr=np.array(errs).T, color=colors, capsize=4)
ax.set_xticks(range(len(names)), [n.replace(', ', '\n') for n in names], fontsize=8)
ax.axhline(0.5, color='gray', ls=':', lw=1)
ax.set_ylabel('AUROC')
ax.set_title(f'Q2: A vs B — paired Δ(B−A) = {D_BA[0]:+.3f} [{D_BA[1]:+.3f}, {D_BA[2]:+.3f}]')

# (4) sample H(n) trajectories per T (first 3 common samples)
ax = axes[1, 1]
cmap = plt.get_cmap('coolwarm')
for ti, T in enumerate(TEMPS):
    cache = load_cache(run_path(T, 0))
    for j, i in enumerate(COMMON_IDX[:3]):
        ents = cache[i]['token_entropies']
        ax.plot(ents, color=cmap(ti / (len(TEMPS) - 1)), alpha=0.6, lw=0.8,
                label=f'T={T}' if j == 0 else None)
ax.set_xlabel('token index'); ax.set_ylabel('H(n)')
ax.set_title('Sample entropy trajectories by temperature')
ax.legend(fontsize=8)

plt.tight_layout()
fig_path = os.path.join(RES_DIR, 'phase15_summary.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Figure saved: {fig_path}')
plt.show()

## HISTORY.md template (fill in after the run)

```
### Step N — Phase 15: temperature variation (Item 6) — results

**What**: 9 runs × 200 MATH-500 samples on Qwen2.5-Math-7B (T ∈ {0.3, 0.6, 1.0, 1.5, 2.0}
+ 4 extra T=1.0). Q1 AUROC-vs-T curve; Q2 paired A/B ablation (same-T vs multi-T K=5 fusion).
**Why**: Item 6 of the Jun 17 meeting; runs also repay the Extension E raw-trace debt
(T=1.0 run0 = canonical MATH-500/Qwen-7B raw cache, top-50 logprobs included).
**Result**: G-T1 <PASS/FAIL> (<numbers>); G-T2 <PASS/FAIL> (Δ(B−A) = <x> [<lo>, <hi>]).
Per-T AUROC: <table>. Accuracy vs T: <numbers>.
```